# ETL Pipeline

In [1]:
import pandas as pd

### Load cleaned CSV

In [2]:
uof = pd.read_csv("cleaned_data.csv")

In [3]:
uof.head()

,ID,Incident_Type,Date_Time,Officer_ID,Subject_ID,Subject_Race,Subject_Gender,Year,Month
0,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1
1,2015UOF-1321-1306-5300,Level 2 - Use of Force,2015-08-03 15:30:00,1615,5261,White,Male,2015,8
2,2020UOF-1310-1097-23017,Level 2 - Use of Force,2020-07-19 15:35:00,1672,23898,Nat Hawaiian/Oth Pac Islander,Male,2020,7
3,2022UOF-1269-2902-29489,Level 2 - Use of Force,2022-11-10 06:29:00,5708,30362,White,Female,2022,11


### Load external dataset

In [4]:
encamp = pd.read_csv("Unauthorized_Encampment_Reports_20260221.csv")

/var/folders/nf/xj9w2p992vzfzsk5h40k9hb40000gn/T/ipykernel_67200/1494922539.py:1: DtypeWarning: Columns (10,14,15,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  encamp = pd.read_csv("Unauthorized_Encampment_Reports_20260221.csv")


In [5]:
encamp.head()

,Service Request Number,Created Date,Method Received,Status,Location,X_Value,Y_Value,Latitude,Longitude,Latitude/Longitude,ZIP Code,Council District,Police Precinct,Community Reporting Area,Are there people present?,"Are there tents, structures, or tarps?",Are there RVs/cars/misc. vehicles?,Is the encampment blocking access?,Is there trash or debris?
0,18-00062016,04/03/2018 11:49:19 AM,Phone,Closed,"7400 SAND POINT WAY NE HILL NEXT TO KITE HILL,...",1.288198e+06,252090.600118,47.682125,-122.263301,POINT (-122.26330105 47.68212498),98115,4.0,NORTH,NaN,NaN,NaN,NaN,NaN,NaN
1,18-00062201,04/03/2018 02:28:10 PM,Phone,Closed,"4434 46TH AVE S, SEATTLE, WA 98118",1.284116e+06,208676.428682,47.563066,-122.275447,POINT (-122.27544656 47.56306595),98118,2.0,SOUTH,NaN,2.0,NaN,NaN,NaN,NaN
2,18-00062281,04/03/2018 03:34:14 PM,Phone,Closed,"15TH AVE NE & NE RAVENNA BLVD, SEATTLE, WA",1.276150e+06,248157.711671,47.670721,-122.311896,POINT (-122.31189583 47.67072083),NaN,4.0,NORTH,NaN,1.0,NaN,NaN,NaN,NaN
3,18-00062298,04/03/2018 03:45:00 PM,Find It Fix It Apps,Closed,"1226 LAKEVIEW BLVD E, SEATTLE, WA 98102",1.272815e+06,233581.630115,47.630592,-122.324283,POINT (-122.3242828 47.63059175),98102,3.0,EAST,NaN,NaN,0.0,0.0,NaN,NaN
4,18-00062310,04/03/2018 03:54:26 PM,Phone,Closed,"3201 FAIRVIEW AVE E, SEATTLE, WA 98102",1.273105e+06,240996.677905,47.650932,-122.323692,POINT (-122.32369211 47.65093156),98102,4.0,WEST,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print("Use of Force rows/cols:", uof.shape)

Use of Force rows/cols: (4, 9)


In [7]:
print("Encampment rows/cols:", encamp.shape)

Encampment rows/cols: (256741, 19)


In [8]:
print("\nUse of Force columns:\n", uof.columns)


Use of Force columns:
 Index(['ID', 'Incident_Type', 'Date_Time', 'Officer_ID', 'Subject_ID',
       'Subject_Race', 'Subject_Gender', 'Year', 'Month'],
      dtype='object')


In [9]:
print("\nEncampment columns:\n", encamp.columns)


Encampment columns:
 Index(['Service Request Number', 'Created Date', 'Method Received ', 'Status',
       'Location', 'X_Value', 'Y_Value', 'Latitude', 'Longitude',
       'Latitude/Longitude', 'ZIP Code', 'Council District', 'Police Precinct',
       'Community Reporting Area', 'Are there people present?',
       'Are there tents, structures, or tarps?',
       'Are there RVs/cars/misc. vehicles?',
       'Is the encampment blocking access?', 'Is there trash or debris?'],
      dtype='object')


### Encampment data origin

This encampment data is from the City of Seattle website: https://data.seattle.gov/City-Administration/Unauthorized-Encampment-Reports/k7ra-jqqe/about_data

The data were pulled on Sat, Feb 21

### Why use encampment data?

Encampment data were chosen to enrich the Use of Force data with community and environment context.

Encampment reports can be joined/compared by geography, and or date/time data, enabling analysis of whether incident patterns align with encampment reporting patterns.

# Transform encampment data

In [10]:
# fix column names
encamp.columns = (
    encamp.columns
    .str.strip()
    .str.lower()
    .str.replace("/", "_", regex=False)
    .str.replace("?", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

In [11]:
encamp.columns

Index(['service_request_number', 'created_date', 'method_received', 'status',
       'location', 'x_value', 'y_value', 'latitude', 'longitude',
       'latitude_longitude', 'zip_code', 'council_district', 'police_precinct',
       'community_reporting_area', 'are_there_people_present',
       'are_there_tents,_structures,_or_tarps',
       'are_there_rvs_cars_misc_vehicles', 'is_the_encampment_blocking_access',
       'is_there_trash_or_debris'],
      dtype='object')

In [13]:
# drop columns and keep ones we'll use
keep_cols = [
    "location",
    "zip_code",
    "police_precinct",
    "community_reporting_area",
    "council_district",
    "are_there_people_present",
    "are_there_tents_structures_or_tarps",
    "are_there_rvs_cars_misc_vehicles",
    "is_the_encampment_blocking_access",
    "is_there_trash_or_debris",
]
encamp = encamp[[c for c in keep_cols if c in encamp.columns]].copy()


In [14]:
encamp.head()

,location,zip_code,police_precinct,community_reporting_area,council_district,are_there_people_present,are_there_rvs_cars_misc_vehicles,is_the_encampment_blocking_access,is_there_trash_or_debris
0,"7400 SAND POINT WAY NE HILL NEXT TO KITE HILL,...",98115,NORTH,NaN,4.0,NaN,NaN,NaN,NaN
1,"4434 46TH AVE S, SEATTLE, WA 98118",98118,SOUTH,NaN,2.0,2.0,NaN,NaN,NaN
2,"15TH AVE NE & NE RAVENNA BLVD, SEATTLE, WA",NaN,NORTH,NaN,4.0,1.0,NaN,NaN,NaN
3,"1226 LAKEVIEW BLVD E, SEATTLE, WA 98102",98102,EAST,NaN,3.0,NaN,0.0,NaN,NaN
4,"3201 FAIRVIEW AVE E, SEATTLE, WA 98102",98102,WEST,NaN,4.0,NaN,NaN,NaN,NaN


We'll merge this dataset later on police precinct.

In [17]:
# drop rows with NA 
encamp = encamp[(encamp['police_precinct'].notna())].copy()

In [18]:
encamp.head()

,location,zip_code,police_precinct,community_reporting_area,council_district,are_there_people_present,are_there_rvs_cars_misc_vehicles,is_the_encampment_blocking_access,is_there_trash_or_debris
0,"7400 SAND POINT WAY NE HILL NEXT TO KITE HILL,...",98115,NORTH,NaN,4.0,NaN,NaN,NaN,NaN
1,"4434 46TH AVE S, SEATTLE, WA 98118",98118,SOUTH,NaN,2.0,2.0,NaN,NaN,NaN
2,"15TH AVE NE & NE RAVENNA BLVD, SEATTLE, WA",NaN,NORTH,NaN,4.0,1.0,NaN,NaN,NaN
3,"1226 LAKEVIEW BLVD E, SEATTLE, WA 98102",98102,EAST,NaN,3.0,NaN,0.0,NaN,NaN
4,"3201 FAIRVIEW AVE E, SEATTLE, WA 98102",98102,WEST,NaN,4.0,NaN,NaN,NaN,NaN


In [19]:
encamp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 256091 entries, 0 to 256740
Data columns (total 9 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   location                           256091 non-null  object 
 1   zip_code                           248933 non-null  object 
 2   police_precinct                    256091 non-null  object 
 3   community_reporting_area           192576 non-null  object 
 4   council_district                   255970 non-null  float64
 5   are_there_people_present           176360 non-null  object 
 6   are_there_rvs_cars_misc_vehicles   175191 non-null  object 
 7   is_the_encampment_blocking_access  172719 non-null  object 
 8   is_there_trash_or_debris           163413 non-null  object 
dtypes: float64(1), object(8)
memory usage: 19.5+ MB


Dropped from 256,741 rows to 160,698 rows after removing NA values

We have no NA values on police precinct.

After considering, we will drop more values.

In [20]:
# drop records with no zip-code or values in people present, RVs/cars/misc, encampment blocking, or trash/debris
encamp = encamp.dropna()

In [21]:
encamp.head()

,location,zip_code,police_precinct,community_reporting_area,council_district,are_there_people_present,are_there_rvs_cars_misc_vehicles,is_the_encampment_blocking_access,is_there_trash_or_debris
56107,"5600 14TH AVE NW, SEATTLE, WA 98107",98107,NORTH,BALLARD,6.0,Yes,No,Other (please explain in the Description),Garbage/Loose litter
56116,"1401 NW 45TH ST, SEATTLE, WA 98107",98107,NORTH,BALLARD,6.0,Unknown,Yes,Recreation amenity,Garbage/Loose litter
56171,"5466 29TH AVE SW, SEATTLE, WA 98126",98126,SOUTHWEST,HIGH POINT,1.0,Unknown,Yes,No blockage,"Garbage/Loose litter,Human waste"
56191,"1144 NW LEARY WAY, SEATTLE, WA 98107",98107,NORTH,BALLARD,6.0,Yes,Unknown,Sidewalk,"Garbage/Loose litter,Needles/Sharps,Hazardous ..."
56197,"1132 N 128TH ST, SEATTLE, WA 98133",98133,NORTH,HALLER LAKE,5.0,Yes,Yes,Sidewalk,Garbage/Loose litter


In [22]:
encamp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 160695 entries, 56107 to 256740
Data columns (total 9 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   location                           160695 non-null  object 
 1   zip_code                           160695 non-null  object 
 2   police_precinct                    160695 non-null  object 
 3   community_reporting_area           160695 non-null  object 
 4   council_district                   160695 non-null  float64
 5   are_there_people_present           160695 non-null  object 
 6   are_there_rvs_cars_misc_vehicles   160695 non-null  object 
 7   is_the_encampment_blocking_access  160695 non-null  object 
 8   is_there_trash_or_debris           160695 non-null  object 
dtypes: float64(1), object(8)
memory usage: 12.3+ MB


We now have a fully clean dataset with no NA values, ready to be merged with our original dataset if needed.

In [24]:
#export clean encampment set to csv
encamp.to_csv("encamp_clean.csv", index = False)

### Merge datasets

We need to add police precinct to our cleaned data for the merge to succeed.

In [28]:
uof["police_precinct"] = ["WEST", "NORTH", "WEST", "EAST"]


In [29]:
uof.head()

,ID,Incident_Type,Date_Time,Officer_ID,Subject_ID,Subject_Race,Subject_Gender,Year,Month,police_precinct
0,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST
1,2015UOF-1321-1306-5300,Level 2 - Use of Force,2015-08-03 15:30:00,1615,5261,White,Male,2015,8,NORTH
2,2020UOF-1310-1097-23017,Level 2 - Use of Force,2020-07-19 15:35:00,1672,23898,Nat Hawaiian/Oth Pac Islander,Male,2020,7,WEST
3,2022UOF-1269-2902-29489,Level 2 - Use of Force,2022-11-10 06:29:00,5708,30362,White,Female,2022,11,EAST


Merge dataset using left join

In [ ]:
# optimization

uof["police_precinct"] = (
    uof["police_precinct"]
    .astype("string")
    .str.strip()
    .str.upper()
    .astype("category")
)

df = (
    uof.set_index("police_precinct")
    .join(encamp.set_index("police_precinct"), how="left")
    .reset_index()
)

In [30]:
df = uof.merge(encamp, on ="police_precinct", how ="left")

In [31]:
df.head()

,ID,Incident_Type,Date_Time,Officer_ID,Subject_ID,Subject_Race,Subject_Gender,Year,Month,police_precinct,location,zip_code,community_reporting_area,council_district,are_there_people_present,are_there_rvs_cars_misc_vehicles,is_the_encampment_blocking_access,is_there_trash_or_debris
0,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"1601 15TH AVE W, SEATTLE, WA 98119",98119,INTERBAY,7.0,Yes,No,Sidewalk,"Garbage/Loose litter,Needles/Sharps,Bulky item..."
1,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"2927 FRANKLIN AVE E, SEATTLE, WA 98102",98102,MONTLAKE/PORTAGE BAY,4.0,Yes,Yes,Sidewalk,"Garbage/Loose litter,Needles/Sharps,Hazardous ..."
2,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"670 S PLUMMER ST, SEATTLE, WA 98134",98134,DUWAMISH/SODO,2.0,Yes,Yes,No blockage,"Garbage/Loose litter,Needles/Sharps,Hazardous ..."
3,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"2936 EASTLAKE AVE E, SEATTLE, WA 98102",98102,MONTLAKE/PORTAGE BAY,4.0,Unknown,Yes,Sidewalk,"Garbage/Loose litter,Needles/Sharps,Hazardous ..."
4,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"2936 EASTLAKE AVE E, SEATTLE, WA 98102",98102,MONTLAKE/PORTAGE BAY,4.0,Yes,No,Other (please explain in the Description),"Garbage/Loose litter,Other (please explain in ..."


We now have a fully merged dataset.

### Create new derived column

In [33]:
df["region"] = df["community_reporting_area"].astype(str) + " " + df["zip_code"].astype(str)

In [34]:
df.head()

,ID,Incident_Type,Date_Time,Officer_ID,Subject_ID,Subject_Race,Subject_Gender,Year,Month,police_precinct,location,zip_code,community_reporting_area,council_district,are_there_people_present,are_there_rvs_cars_misc_vehicles,is_the_encampment_blocking_access,is_there_trash_or_debris,region
0,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"1601 15TH AVE W, SEATTLE, WA 98119",98119,INTERBAY,7.0,Yes,No,Sidewalk,"Garbage/Loose litter,Needles/Sharps,Bulky item...",INTERBAY 98119
1,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"2927 FRANKLIN AVE E, SEATTLE, WA 98102",98102,MONTLAKE/PORTAGE BAY,4.0,Yes,Yes,Sidewalk,"Garbage/Loose litter,Needles/Sharps,Hazardous ...",MONTLAKE/PORTAGE BAY 98102
2,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"670 S PLUMMER ST, SEATTLE, WA 98134",98134,DUWAMISH/SODO,2.0,Yes,Yes,No blockage,"Garbage/Loose litter,Needles/Sharps,Hazardous ...",DUWAMISH/SODO 98134
3,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"2936 EASTLAKE AVE E, SEATTLE, WA 98102",98102,MONTLAKE/PORTAGE BAY,4.0,Unknown,Yes,Sidewalk,"Garbage/Loose litter,Needles/Sharps,Hazardous ...",MONTLAKE/PORTAGE BAY 98102
4,2015UOF-0072-1204-3004,Level 2 - Use of Force,2015-01-19 15:00:00,1634,2984,White,Male,2015,1,WEST,"2936 EASTLAKE AVE E, SEATTLE, WA 98102",98102,MONTLAKE/PORTAGE BAY,4.0,Yes,No,Other (please explain in the Description),"Garbage/Loose litter,Other (please explain in ...",MONTLAKE/PORTAGE BAY 98102


We now have a new variable - region - based on zip code and community reporting area.

In [35]:
# export to pdf
df.to_csv("team2_final_data.csv", index = False)